In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import pickle
from collections import Counter
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import classification_report
import seaborn as sns

# Load data splits
with open('data/processed/data_splits.pkl', 'rb') as f:
    data = pickle.load(f)

train_paths = data['train_paths']
train_labels = data['train_labels']
val_paths = data['val_paths']
val_labels = data['val_labels']
test_paths = data['test_paths']
test_labels = data['test_labels']
classes = data['classes']

print("Data loaded!")
print(f"Classes: {len(classes)}")
print(f"\nClass distribution in training set:")
label_counts = Counter(train_labels)
for idx, count in sorted(label_counts.items(), key=lambda x: x[1]):
    bar = '█' * (count // 50)
    print(f"  {count:5d} {bar} {classes[idx]}")

Data loaded!
Classes: 15

Class distribution in training set:
    106 ██ Potato___healthy
    261 █████ Tomato__Tomato_mosaic_virus
    667 █████████████ Tomato_Leaf_Mold
    699 █████████████ Pepper__bell___Bacterial_spot
    700 ██████████████ Potato___Late_blight
    700 ██████████████ Potato___Early_blight
    700 ██████████████ Tomato_Early_blight
    983 ███████████████████ Tomato__Target_Spot
   1035 ████████████████████ Pepper__bell___healthy
   1114 ██████████████████████ Tomato_healthy
   1174 ███████████████████████ Tomato_Spider_mites_Two_spotted_spider_mite
   1240 ████████████████████████ Tomato_Septoria_leaf_spot
   1337 ██████████████████████████ Tomato_Late_blight
   1490 █████████████████████████████ Tomato_Bacterial_spot
   2247 ████████████████████████████████████████████ Tomato__Tomato_YellowLeaf__Curl_Virus


In [2]:
# Calculate weight for each sample
# Rare classes get higher weights so they're sampled more often
class_counts = Counter(train_labels)
total_samples = len(train_labels)

# Weight for each class = total / count (rare classes get higher weight)
class_weights = {cls: total_samples / count 
                 for cls, count in class_counts.items()}

print("Class weights (higher = sampled more often):")
for idx, weight in sorted(class_weights.items(), key=lambda x: x[1], reverse=True):
    print(f"  {weight:6.1f}x  {classes[idx]}")

# Create sample weights array
sample_weights = [class_weights[label] for label in train_labels]
sample_weights = torch.tensor(sample_weights, dtype=torch.float32)

# Create sampler
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print(f"\nSampler created for {len(sample_weights)} training samples")

Class weights (higher = sampled more often):
   136.3x  Potato___healthy
    55.4x  Tomato__Tomato_mosaic_virus
    21.7x  Tomato_Leaf_Mold
    20.7x  Pepper__bell___Bacterial_spot
    20.6x  Potato___Late_blight
    20.6x  Potato___Early_blight
    20.6x  Tomato_Early_blight
    14.7x  Tomato__Target_Spot
    14.0x  Pepper__bell___healthy
    13.0x  Tomato_healthy
    12.3x  Tomato_Spider_mites_Two_spotted_spider_mite
    11.7x  Tomato_Septoria_leaf_spot
    10.8x  Tomato_Late_blight
     9.7x  Tomato_Bacterial_spot
     6.4x  Tomato__Tomato_YellowLeaf__Curl_Virus

Sampler created for 14453 training samples


In [3]:
class PlantDiseaseDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = PlantDiseaseDataset(train_paths, train_labels, transform=train_transform)
val_dataset   = PlantDiseaseDataset(val_paths,   val_labels,   transform=val_transform)
test_dataset  = PlantDiseaseDataset(test_paths,  test_labels,  transform=val_transform)

# Use sampler instead of shuffle=True
train_loader = DataLoader(train_dataset, batch_size=32,
                          sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=32,
                          shuffle=False,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=32,
                          shuffle=False,  num_workers=0)

print("DataLoaders created with WeightedRandomSampler!")
print(f"Train batches: {len(train_loader)}")

# Verify sampler is working — check class distribution in first 500 batches
sample_labels = []
for i, (_, labels) in enumerate(train_loader):
    sample_labels.extend(labels.numpy())
    if i >= 15: break

sampled_counts = Counter(sample_labels)
print(f"\nClass distribution after sampling (first 16 batches):")
for idx in range(len(classes)):
    count = sampled_counts.get(idx, 0)
    bar = '█' * (count // 2)
    print(f"  {count:4d} {bar} {classes[idx]}")

DataLoaders created with WeightedRandomSampler!
Train batches: 452

Class distribution after sampling (first 16 batches):
    38 ███████████████████ Pepper__bell___Bacterial_spot
    32 ████████████████ Pepper__bell___healthy
    32 ████████████████ Potato___Early_blight
    34 █████████████████ Potato___Late_blight
    34 █████████████████ Potato___healthy
    28 ██████████████ Tomato_Bacterial_spot
    28 ██████████████ Tomato_Early_blight
    31 ███████████████ Tomato_Late_blight
    43 █████████████████████ Tomato_Leaf_Mold
    45 ██████████████████████ Tomato_Septoria_leaf_spot
    54 ███████████████████████████ Tomato_Spider_mites_Two_spotted_spider_mite
    28 ██████████████ Tomato__Target_Spot
    27 █████████████ Tomato__Tomato_YellowLeaf__Curl_Virus
    28 ██████████████ Tomato__Tomato_mosaic_virus
    30 ███████████████ Tomato_healthy


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load best model from Day 5
model = models.resnet50(weights=None)
model.fc = nn.Linear(2048, len(classes))
model.load_state_dict(torch.load('models/resnet50_finetuned.pth', map_location='cpu'))
model = model.to(device)

# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

def evaluate(model, loader, criterion, device):
    model.eval()
    correct, total, running_loss = 0, 0, 0.0
    all_preds, all_true = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_true.extend(labels.cpu().numpy())
    return running_loss/len(loader), 100.*correct/total, all_preds, all_true

best_val_acc = 0.0
num_epochs = 3

print("Fine-tuning with weighted sampling...\n")

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    print("-" * 40)

    model.train()
    correct, total, running_loss = 0, 0, 0.0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        if (batch_idx + 1) % 50 == 0:
            print(f"  Batch {batch_idx+1}/{len(train_loader)} "
                  f"| Loss: {running_loss/(batch_idx+1):.3f} "
                  f"| Acc: {100.*correct/total:.1f}%")

    train_acc = 100.*correct/total
    val_loss, val_acc, val_preds, val_true = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'models/resnet50_balanced.pth')
        print(f"  ✓ New best model saved!")

    print(f"\n  Train Acc: {train_acc:.1f}% | Val Acc: {val_acc:.1f}%\n")

print(f"Done! Best val accuracy: {best_val_acc:.1f}%")

Fine-tuning with weighted sampling...

Epoch 1/3
----------------------------------------
  Batch 50/452 | Loss: 0.172 | Acc: 94.8%
  Batch 100/452 | Loss: 0.186 | Acc: 94.5%
  Batch 150/452 | Loss: 0.173 | Acc: 94.8%
  Batch 200/452 | Loss: 0.155 | Acc: 95.4%
  Batch 250/452 | Loss: 0.145 | Acc: 95.5%
  Batch 300/452 | Loss: 0.135 | Acc: 95.8%
  Batch 350/452 | Loss: 0.129 | Acc: 96.1%
  Batch 400/452 | Loss: 0.124 | Acc: 96.1%
  Batch 450/452 | Loss: 0.117 | Acc: 96.3%
  ✓ New best model saved!

  Train Acc: 96.3% | Val Acc: 98.2%

Epoch 2/3
----------------------------------------
  Batch 50/452 | Loss: 0.104 | Acc: 96.7%
  Batch 100/452 | Loss: 0.096 | Acc: 96.9%
  Batch 150/452 | Loss: 0.089 | Acc: 97.1%
  Batch 200/452 | Loss: 0.090 | Acc: 96.9%
  Batch 250/452 | Loss: 0.087 | Acc: 97.0%
  Batch 300/452 | Loss: 0.085 | Acc: 97.1%
  Batch 350/452 | Loss: 0.084 | Acc: 97.2%
  Batch 400/452 | Loss: 0.082 | Acc: 97.2%
  Batch 450/452 | Loss: 0.080 | Acc: 97.3%
  ✓ New best model save

In [5]:
# Evaluate balanced model on test set
model.load_state_dict(torch.load('models/resnet50_balanced.pth'))
_, test_acc, test_preds, test_true = evaluate(model, test_loader, criterion, device)

print(f"Balanced model test accuracy: {test_acc:.2f}%\n")
print("Per-class breakdown:")
print(classification_report(test_true, test_preds, target_names=classes))

Balanced model test accuracy: 99.74%

Per-class breakdown:
                                             precision    recall  f1-score   support

              Pepper__bell___Bacterial_spot       0.99      1.00      1.00       149
                     Pepper__bell___healthy       1.00      0.99      1.00       222
                      Potato___Early_blight       1.00      1.00      1.00       150
                       Potato___Late_blight       0.99      1.00      1.00       150
                           Potato___healthy       1.00      1.00      1.00        23
                      Tomato_Bacterial_spot       1.00      1.00      1.00       319
                        Tomato_Early_blight       0.97      0.99      0.98       150
                         Tomato_Late_blight       1.00      0.99      0.99       286
                           Tomato_Leaf_Mold       1.00      1.00      1.00       143
                  Tomato_Septoria_leaf_spot       1.00      1.00      1.00       266
Tomat

In [1]:
new_api = open('src/api.py').read().replace(
    'resnet.load_state_dict(torch.load("models/resnet50_finetuned.pth",',
    'resnet.load_state_dict(torch.load("models/resnet50_balanced.pth",'
)

with open('src/api.py', 'w') as f:
    f.write(new_api)

print("API updated to use balanced model!")

API updated to use balanced model!
